# Module 03 — QC Overview Across Datasets

This notebook provides a cross-dataset quality-control summary for the IVD scRNA-seq atlas.
We compiled **12 publicly available datasets** spanning nucleus pulposus, annulus fibrosus,
cartilaginous endplate, and mixed IVD tissues. Each dataset was independently preprocessed
through a uniform pipeline (`scripts/03_preprocessing.py`) that applies:

1. **Cell-level QC** — filtering on minimum genes detected, maximum mitochondrial fraction, and doublet removal (Scrublet)
2. **Normalization & HVG selection** — library-size normalization, log-transformation, and selection of 3,000 highly variable genes
3. **Dimensionality reduction** — PCA (50 components), UMAP embedding, and Leiden clustering at multiple resolutions
4. **Preliminary cell-type annotation** — marker-gene scoring against canonical IVD and stromal/immune signatures

The goal here is to **compare QC outcomes across studies** before integration (Module 04),
flagging any datasets with unusual dropout, sequencing depth, or cell-type composition.

**Manuscript mapping:** Supplementary Figure S1 — QC metrics across datasets. Methods section on preprocessing.

**Data sources:**
- `data/processed/{accession}.h5ad` — preprocessed AnnData objects (12 files)
- `results/qc_reports/qc_summary.tsv` — per-dataset QC summary statistics

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='scanpy')

from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('..').resolve()
PROC_DIR = BASE / 'data' / 'processed'
QC_DIR = BASE / 'results' / 'qc_reports'
QC_DIR.mkdir(parents=True, exist_ok=True)

ALL_ACCESSIONS = [
    'GSE160756', 'GSE165722', 'GSE189916', 'GSE199866', 'GSE205535',
    'CNP0002664', 'GSE233666', 'GSE244889', 'GSE251686', 'GSE255768',
    'GSE230809', 'GSE242443',
]

# Consistent palette across studies
PALETTE = dict(zip(ALL_ACCESSIONS, sns.color_palette('tab20', len(ALL_ACCESSIONS))))

# ── Load qc_summary.tsv ───────────────────────────────────────────────────
qc_summary_path = QC_DIR / 'qc_summary.tsv'
qc_df = pd.read_csv(qc_summary_path, sep='\t') if qc_summary_path.exists() else pd.DataFrame()
if not qc_df.empty:
    qc_df = qc_df.set_index('accession')
print(f'qc_summary.tsv: {len(qc_df)} datasets')

# ── Load obs DataFrames (no .X) ───────────────────────────────────────────
obs_frames = {}
for acc in ALL_ACCESSIONS:
    path = PROC_DIR / f'{acc}.h5ad'
    if not path.exists():
        print(f'  SKIP {acc}: file not found')
        continue
    adata = sc.read_h5ad(path, backed='r')
    obs_frames[acc] = adata.obs.to_memory() if hasattr(adata.obs, 'to_memory') else adata.obs.copy()
    adata.file.close()
    print(f'  {acc}: {len(obs_frames[acc]):,} cells')

print(f'\nLoaded {len(obs_frames)} datasets')

## Summary Table

The table below shows key QC metrics for each dataset: how many cells we started with,
how many survived filtering, and the per-cell distributions of gene detection and UMI counts.

- **Cells (raw)** — total barcodes before any filtering (from `qc_summary.tsv`; "—" if the dataset was added after the initial QC run).
- **Retention (%)** — fraction of raw barcodes kept after QC. Low retention can indicate poor-quality samples or aggressive filtering.
- **Median genes/cell** and **Median counts/cell** — proxies for library complexity and sequencing depth, respectively.
- **Median %MT** — mitochondrial fraction; elevated values suggest damaged or stressed cells.

In [ ]:
rows = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    cells_after = len(obs)
    cells_raw = int(qc_df.loc[acc, 'cells_raw_total']) if acc in qc_df.index else np.nan
    retention = qc_df.loc[acc, 'retention_pct'] if acc in qc_df.index else np.nan
    rows.append({
        'Accession': acc,
        'Cells (raw)': int(cells_raw) if not np.isnan(cells_raw) else '—',
        'Cells (after QC)': cells_after,
        'Retention (%)': f'{retention:.1f}' if not np.isnan(retention) else '—',
        'Median genes/cell': int(obs['n_genes_by_counts'].median()),
        'Median counts/cell': int(obs['total_counts'].median()),
        'Median %MT': f"{obs['pct_counts_mt'].median():.2f}",
    })

summary_df = pd.DataFrame(rows)

# Totals row
total_raw = sum(r['Cells (raw)'] for r in rows if isinstance(r['Cells (raw)'], int))
total_after = summary_df['Cells (after QC)'].sum()
totals = {
    'Accession': 'TOTAL',
    'Cells (raw)': total_raw,
    'Cells (after QC)': total_after,
    'Retention (%)': '',
    'Median genes/cell': '',
    'Median counts/cell': '',
    'Median %MT': '',
}
summary_df = pd.concat([summary_df, pd.DataFrame([totals])], ignore_index=True)

display(summary_df.style.set_caption('QC Summary — All Datasets').hide(axis='index'))

## Cross-Dataset QC Distributions

The violin plots below show the **per-cell distributions** of three core QC metrics across
all 12 datasets. Each violin spans the full range of values for that dataset, with an
embedded box plot showing the median and interquartile range.

**What to look for:**
- **Genes detected** (top panel): Datasets with very low median gene counts may have shallow sequencing or low-complexity libraries. Large differences here will affect downstream integration — datasets with consistently fewer genes detected will contribute fewer informative features.
- **Total counts / UMIs** (middle panel): Reflects sequencing depth per cell. Platforms and protocols vary widely (e.g., 10x Chromium vs. Smart-seq2 vs. microwell-based), so some spread is expected. Extreme outliers within a dataset may indicate ambient RNA contamination or multiplets that escaped Scrublet.
- **Mitochondrial fraction** (bottom panel): High %MT often indicates dying or mechanically damaged cells. Our QC threshold was set at 20%, but most cells should be well below this. Datasets derived from surgical tissue may show higher baseline %MT than those from cell culture.

In [ ]:
# Build long-form DataFrame for violin plots
qc_long = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    sub = obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].copy()
    sub['dataset'] = acc
    qc_long.append(sub)
qc_long = pd.concat(qc_long, ignore_index=True)

metrics = [
    ('n_genes_by_counts', 'Genes detected', None),
    ('total_counts', 'Total counts (UMI)', None),
    ('pct_counts_mt', '% mitochondrial counts', (0, 15)),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), constrained_layout=True)
for ax, (col, label, ylim) in zip(axes, metrics):
    sns.violinplot(
        data=qc_long, x='dataset', y=col, order=ALL_ACCESSIONS,
        palette=PALETTE, inner='box', linewidth=0.5, cut=0, ax=ax,
        density_norm='width', scale='width' if hasattr(sns, '__version__') else None,
    )
    ax.set_ylabel(label)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
    if ylim:
        ax.set_ylim(ylim)
fig.suptitle('Cross-Dataset QC Distributions', fontsize=14, y=1.01)

fig.savefig(QC_DIR / 'notebook_03_qc_violins.png')
plt.show()

## Cells Retained vs Removed

The stacked bar chart below shows the absolute number of cells **retained** (blue) versus
**removed** (orange) during QC for each dataset, with the overall retention percentage
annotated above each bar.

Cell removal is driven by three filters applied sequentially:
1. Low-quality cells with too few genes detected
2. Cells exceeding the mitochondrial fraction threshold
3. Predicted doublets (Scrublet)

Datasets with retention below ~80% warrant closer inspection — this may reflect genuine
sample quality issues (e.g., tissue dissociation damage) or overly aggressive thresholds
for that particular platform. Conversely, very high retention (>95%) suggests clean
capture with minimal dead-cell contamination.

In [ ]:
# Build data for stacked bar chart
bar_data = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    cells_after = len(obs_frames[acc])
    if acc in qc_df.index:
        cells_raw = int(qc_df.loc[acc, 'cells_raw_total'])
        cells_removed = cells_raw - cells_after
    else:
        cells_raw = np.nan
        cells_removed = np.nan
    bar_data.append({'dataset': acc, 'Retained': cells_after, 'Removed': cells_removed})

bar_df = pd.DataFrame(bar_data).set_index('dataset')
# Drop datasets with no raw count info
bar_df_plot = bar_df.dropna()

fig, ax = plt.subplots(figsize=(12, 5))
bar_df_plot[['Retained', 'Removed']].plot.bar(
    stacked=True, ax=ax, color=['#4c72b0', '#dd8452'], edgecolor='white', width=0.7,
)
ax.set_ylabel('Number of cells')
ax.set_xlabel('')
ax.set_title('Cells Retained vs Removed per Dataset')
ax.tick_params(axis='x', rotation=45)
ax.legend(frameon=False)

# Annotate retention %
for i, (idx, row) in enumerate(bar_df_plot.iterrows()):
    total = row['Retained'] + row['Removed']
    pct = row['Retained'] / total * 100
    ax.text(i, total + total * 0.01, f'{pct:.0f}%', ha='center', va='bottom', fontsize=8)

fig.tight_layout()
fig.savefig(QC_DIR / 'notebook_03_cells_retained.png')
plt.show()

## Sequencing Depth Comparison

Sequencing depth varies substantially across studies due to differences in platform, protocol,
and sequencing budget. This section provides two complementary views:

- **Panel A (left)** — Box plots of per-cell total UMI counts for each dataset. The box shows
  the interquartile range (IQR) and median; whiskers extend to 1.5x IQR. This reveals both the
  typical sequencing depth and how variable it is within each study.

- **Panel B (right)** — A scatter plot of **median genes vs. median counts** per dataset.
  Datasets in the upper-right corner have both high sequencing depth and high gene detection
  (ideal). Datasets that fall below the main trend may have lower library complexity despite
  adequate sequencing, which can happen with certain tissue types or capture platforms.

These depth differences are important context for integration: shallower datasets may
under-represent rare transcripts, and batch correction methods need to account for
systematic depth differences.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel A: Box plot of total_counts per dataset
ax = axes[0]
sns.boxplot(
    data=qc_long, x='dataset', y='total_counts', order=ALL_ACCESSIONS,
    palette=PALETTE, fliersize=0.5, linewidth=0.6, ax=ax,
)
ax.set_ylabel('Total counts (UMI per cell)')
ax.set_xlabel('')
ax.set_title('A. Sequencing depth per dataset')
ax.tick_params(axis='x', rotation=45)

# Panel B: Median genes vs median counts per dataset
ax = axes[1]
med_stats = []
for acc in ALL_ACCESSIONS:
    if acc not in obs_frames:
        continue
    obs = obs_frames[acc]
    med_stats.append({
        'dataset': acc,
        'median_genes': obs['n_genes_by_counts'].median(),
        'median_counts': obs['total_counts'].median(),
    })
med_df = pd.DataFrame(med_stats)
for _, row in med_df.iterrows():
    ax.scatter(row['median_counts'], row['median_genes'],
               color=PALETTE[row['dataset']], s=80, zorder=3)
    ax.annotate(row['dataset'], (row['median_counts'], row['median_genes']),
                fontsize=7, ha='left', va='bottom', xytext=(4, 4),
                textcoords='offset points')
ax.set_xlabel('Median total counts per cell')
ax.set_ylabel('Median genes per cell')
ax.set_title('B. Median genes vs counts per dataset')

fig.tight_layout()
fig.savefig(QC_DIR / 'notebook_03_seq_depth.png')
plt.show()

## Per-Dataset UMAP Panels

The grid below shows UMAP embeddings for each dataset individually (one row per dataset,
three columns). These are computed **before cross-dataset integration**, so each UMAP
reflects only the internal structure of that study.

**Columns:**
- **Sample** (left) — cells colored by sample/donor of origin. Good mixing across the UMAP
  suggests low batch effects within the study; strong sample separation may indicate
  biological differences (e.g., degeneration grade) or technical batch effects.
- **Leiden cluster** (center) — unsupervised clusters at resolution 0.5. This gives a sense
  of how many transcriptionally distinct populations each dataset contains.
- **Preliminary cell type** (right) — marker-gene-based labels. These are provisional
  annotations used for sanity-checking before integration; final cell-type assignments will
  be refined in Module 04.

**What to look for:**
- Do the expected IVD cell types (NP cells, AF fibroblasts, endplate chondrocytes) appear?
- Are immune populations (macrophages, T cells) present, and if so, are they concentrated
  in particular samples (e.g., degenerated discs)?
- Are there orphan clusters with no clear cell-type assignment that may need manual inspection?

In [ ]:
# Load UMAP embeddings for each dataset
umap_data = {}
for acc in ALL_ACCESSIONS:
    path = PROC_DIR / f'{acc}.h5ad'
    if not path.exists():
        continue
    adata = sc.read_h5ad(path, backed='r')
    umap_data[acc] = {
        'umap': adata.obsm['X_umap'][:],  # copy to memory
        'sample_id': obs_frames[acc]['sample_id'].values,
        'leiden': obs_frames[acc]['leiden_res_0.5'].astype(str).values,
        'cell_type': obs_frames[acc]['cell_type_preliminary'].values,
    }
    adata.file.close()

datasets = [acc for acc in ALL_ACCESSIONS if acc in umap_data]
n_datasets = len(datasets)
color_keys = ['sample_id', 'leiden', 'cell_type']
col_titles = ['Sample', 'Leiden cluster', 'Preliminary cell type']

fig, axes = plt.subplots(n_datasets, 3, figsize=(18, 4 * n_datasets))
if n_datasets == 1:
    axes = axes[np.newaxis, :]

for row_i, acc in enumerate(datasets):
    umap = umap_data[acc]['umap']
    for col_i, (key, title) in enumerate(zip(color_keys, col_titles)):
        ax = axes[row_i, col_i]
        labels = umap_data[acc][key]
        unique_labels = sorted(set(labels), key=str)
        n_colors = len(unique_labels)
        cpal = sns.color_palette('tab20', n_colors) if n_colors <= 20 else sns.color_palette('husl', n_colors)
        color_map = dict(zip(unique_labels, cpal))

        # Shuffle points for better overlap rendering
        idx = np.random.RandomState(42).permutation(len(umap))
        colors = [color_map[labels[i]] for i in idx]
        ax.scatter(umap[idx, 0], umap[idx, 1], c=colors, s=0.5, alpha=0.6, rasterized=True)
        ax.set_xticks([])
        ax.set_yticks([])
        if row_i == 0:
            ax.set_title(title, fontsize=11)
        if col_i == 0:
            ax.set_ylabel(acc, fontsize=10, fontweight='bold')

fig.suptitle('Per-Dataset UMAP Panels', fontsize=14, y=1.0)
fig.tight_layout()
fig.savefig(QC_DIR / 'notebook_03_umap_panels.png', dpi=150)
plt.show()

## Notes and Next Steps

**QC parameters used across all datasets:**
- Minimum genes per cell: 200
- Maximum mitochondrial fraction: 20%
- Doublet detection: Scrublet (automatic threshold)
- HVGs: 3,000 per dataset (flavor: `seurat_v3`)
- PCA: 50 components; Leiden clustering at resolutions 0.2, 0.5, 0.8, 1.0, 1.5

**Caveats:**
- Three datasets (GSE199866, GSE205535, GSE242443) were added after the initial
  batch QC run and are missing raw cell counts in `qc_summary.tsv` (shown as "—").
  Their post-QC metrics are computed directly from the `.h5ad` files.
- Preliminary cell-type labels are based on marker-gene scoring and should be treated
  as provisional. Definitive annotation will follow in Module 04 after cross-dataset
  integration and reference-based label transfer.

**Figures saved** to `results/qc_reports/` with the `notebook_03_` prefix for
inclusion in the manuscript supplementary materials.

In [ ]:
saved_figures = sorted(QC_DIR.glob('notebook_03_*.png'))
print('Saved figures:')
for f in saved_figures:
    print(f'  {f.relative_to(BASE)}')